# simprep 01: prep

Applies a fully decided manifest to its structure: prepared system file(s) (mmCIF + PDB),
`prep_record.json` and `prep_report.md`, in a run directory on Drive
(`<drive_root>/<yymm>/run_<yymmdd_hhmmss>/`, with `config.json`).

1. Edit the config cell (the only place with settings).
2. Run all cells. To resume an interrupted run, set `run_dir` to that run's directory.

With `demo = True` the notebook preps the committed 6OIM panel structure with its test
fixture decisions and writes under `googleColab_test/`.

In [ ]:
# All configuration lives here (CLAUDE.md, Colab convention 3).
demo = True
CONFIG = {
    "repo_url": "https://github.com/dhavirus/proteinMD_pipeline.git",
    "repo_commit": "SET_A_COMMIT_SHA",  # pinned simprep commit to run
    "drive_root": "/content/drive/MyDrive/"
    + ("googleColab_test" if demo else "googleColab_run")
    + "/simprep",
    "work_root": "/content/simprep_work",
    "structure": "/content/drive/MyDrive/simprep_inputs/structure.cif.gz",
    "manifest": "/content/drive/MyDrive/simprep_inputs/manifest.json",
    "run_dir": None,  # set to an existing run directory to resume it
    "seed": 20260926,
}
REPO_DIR = "/content/proteinMD_pipeline"

In [ ]:
import os
import subprocess
import sys

if len(CONFIG["repo_commit"]) != 40:
    raise ValueError("set CONFIG['repo_commit'] to the full 40-character commit SHA to run")
from google.colab import drive

drive.mount("/content/drive")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", CONFIG["repo_url"], REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", CONFIG["repo_commit"]], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", CONFIG["repo_commit"]], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
import importlib
from pathlib import Path

import modules.prep_module as prep_module

importlib.reload(prep_module)

if demo:
    import yaml

    from tests.prep_helpers import decided_manifest

    panel = Path(REPO_DIR) / "tests" / "panel"
    fixture = yaml.safe_load((panel / "prep_fixtures" / "6OIM.yaml").read_text())
    CONFIG["structure"] = str(panel / "6OIM.cif.gz")
    manifest_path = Path("/content/6OIM.manifest.json")
    decided_manifest(panel / "6OIM.cif.gz", fixture["decisions"], manifest_path)
    CONFIG["manifest"] = str(manifest_path)

In [ ]:
result = prep_module.run_prep_job(CONFIG)
print(("skipped (outputs complete): " if result.skipped else "written: ") + str(result.run_dir))
print((result.run_dir / "prep_report.md").read_text())